<a href="https://colab.research.google.com/github/ernikurniawati-cmyk/pinkmart-web/blob/main/erni_kurniawati_uts_projek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install Flask Flask-Login Flask-WTF

In [ ]:
import os
import sqlite3
import threading
import time
import secrets
import io

from flask import (
    Flask, request, redirect, url_for,
    render_template_string, flash, Response
)
from flask_login import (
    LoginManager, UserMixin, login_user,
    login_required, logout_user, current_user
)
from flask_wtf.csrf import CSRFProtect
from werkzeug.security import generate_password_hash, check_password_hash
from IPython.display import display, HTML
from google.colab import output
import pandas as pd

# =========================================================
# 1. KONFIGURASI APLIKASI
# =========================================================

app = Flask(__name__)
app.config["SECRET_KEY"] = secrets.token_hex(32)
app.config["SESSION_COOKIE_HTTPONLY"] = True
app.config["SESSION_COOKIE_SAMESITE"] = "Lax"

csrf = CSRFProtect(app)

login_manager = LoginManager()
login_manager.init_app(app)
login_manager.login_view = "login"

DB_PATH = "/content/pinkmart_admin.db"


# =========================================================
# 2. DATABASE SQLITE
# =========================================================

def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    with get_db() as conn:
        # Tabel Admins
        conn.execute("""
            CREATE TABLE IF NOT EXISTS admins (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                password_hash TEXT NOT NULL
            )
        """)

        # Tabel Data Kategori / Jenis Produk
        conn.execute("""
            CREATE TABLE IF NOT EXISTS kategori (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nama_kategori TEXT UNIQUE NOT NULL,
                deskripsi TEXT
            )
        """)

        # Tabel Data Barang / Produk
        conn.execute("""
            CREATE TABLE IF NOT EXISTS barang (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                kode_barang TEXT UNIQUE NOT NULL,
                nama_barang TEXT NOT NULL,
                kategori TEXT NOT NULL,
                harga REAL NOT NULL,
                stok INTEGER NOT NULL
            )
        """)

        # Tabel Data Penjualan / Pesanan
        conn.execute("""
            CREATE TABLE IF NOT EXISTS penjualan (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                kode_trx TEXT NOT NULL,
                nama_pelanggan TEXT NOT NULL,
                kategori TEXT NOT NULL DEFAULT 'Elektronik',
                produk TEXT NOT NULL,
                jumlah INTEGER NOT NULL,
                total_harga REAL NOT NULL,
                tanggal TEXT NOT NULL,
                status TEXT NOT NULL
            )
        """)

        # Tabel Data Pelanggan
        conn.execute("""
            CREATE TABLE IF NOT EXISTS pelanggan (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                kode_pelanggan TEXT UNIQUE NOT NULL,
                nama TEXT NOT NULL,
                email TEXT NOT NULL,
                telepon TEXT NOT NULL,
                alamat TEXT NOT NULL,
                status_pelanggan TEXT NOT NULL DEFAULT 'Reguler',
                tanggal_daftar TEXT NOT NULL
            )
        """)

        # Tabel Pengaturan Sistem
        conn.execute("""
            CREATE TABLE IF NOT EXISTS settings (
                id INTEGER PRIMARY KEY CHECK (id = 1),
                nama_aplikasi TEXT NOT NULL,
                versi TEXT NOT NULL,
                mata_uang_default TEXT NOT NULL DEFAULT 'Rp',
                status_portal TEXT NOT NULL DEFAULT 'Aktif',
                footer_text TEXT NOT NULL
            )
        """)

        # Akun Admin Default -> username: pinkmart | password: adminpinkmart
        conn.execute("DELETE FROM admins")
        conn.execute(
            "INSERT INTO admins (username, password_hash) VALUES (?, ?)",
            ("pinkmart", generate_password_hash("adminpinkmart"))
        )

        # Settings Default
        existing_set = conn.execute("SELECT id FROM settings WHERE id = 1").fetchone()
        if existing_set is None:
            conn.execute("""
                INSERT INTO settings (id, nama_aplikasi, versi, mata_uang_default, status_portal, footer_text)
                VALUES (1, 'Web Admin Pinkmart', 'v2.5 Pro', 'Rp', 'Aktif', 'Pinkmart Admin Portal © 2026')
            """)

        # Jenis / Kategori Produk yang Dijual
        conn.execute("DELETE FROM kategori")
        kat_samples = [
            ('Elektronik', 'Gadget, Laptop, Monitor, dan Perangkat Komputer'),
            ('Aksesoris', 'Mouse, Keyboard, Headset, Kabel, dan peripheral pendukung'),
            ('Software', 'Lisensi aplikasi, sistem operasi, dan perangkat lunak'),
            ('Layanan', 'Jasa instalasi, perbaikan, dan servis teknis')
        ]
        conn.executemany("INSERT INTO kategori (nama_kategori, deskripsi) VALUES (?, ?)", kat_samples)

        # Barang Default berdasarkan Jenis Produk
        existing_barang = conn.execute("SELECT id FROM barang").fetchone()
        if existing_barang is None:
            barang_samples = [
                ('BRG-001', 'Laptop Asus Zenbook', 'Elektronik', 14500000, 15),
                ('BRG-002', 'Mouse Wireless Logistics', 'Aksesoris', 225000, 50),
                ('BRG-003', 'Keyboard Mechanical RGB', 'Aksesoris', 850000, 30),
                ('BRG-004', 'Monitor LG 24 Inch', 'Elektronik', 1500000, 20),
            ]
            conn.executemany("INSERT INTO barang (kode_barang, nama_barang, kategori, harga, stok) VALUES (?, ?, ?, ?, ?)", barang_samples)

        # Penjualan Default dengan Pelanggan Baru
        conn.execute("DELETE FROM penjualan")
        sales_samples = [
            ('#TRX-001', 'Gilang Faturahman', 'Elektronik', 'Laptop Asus Zenbook', 1, 14500000, '2026-03-25', 'Selesai'),
            ('#TRX-002', 'Erni Kurniawati', 'Aksesoris', 'Mouse Wireless Logistics', 2, 450000, '2026-03-26', 'Selesai'),
            ('#TRX-003', 'Leon S. Kennedy', 'Aksesoris', 'Keyboard Mechanical RGB', 1, 850000, '2026-03-27', 'Pending'),
        ]
        conn.executemany("""
            INSERT INTO penjualan (kode_trx, nama_pelanggan, kategori, produk, jumlah, total_harga, tanggal, status)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, sales_samples)

        # Pelanggan Default (Gilang Faturahman, Erni Kurniawati, Leon S. Kennedy)
        conn.execute("DELETE FROM pelanggan")
        cust_samples = [
            ('CUST-001', 'Gilang Faturahman', 'gilang@gmail.com', '081234567890', 'Tangerang', 'Pelanggan Setia', '2026-01-10'),
            ('CUST-002', 'Erni Kurniawati', 'erni@yahoo.com', '089876543210', 'Jakarta', 'Pelanggan Setia', '2026-02-14'),
            ('CUST-003', 'Leon S. Kennedy', 'leon@raccoon.com', '085611223344', 'Bandung', 'Reguler', '2026-03-01'),
        ]
        conn.executemany("""
            INSERT INTO pelanggan (kode_pelanggan, nama, email, telepon, alamat, status_pelanggan, tanggal_daftar)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, cust_samples)


init_db()


# =========================================================
# 3. MODEL ADMIN FLASK-LOGIN
# =========================================================

class Admin(UserMixin):
    def __init__(self, row):
        self.id = str(row["id"])
        self.username = row["username"]
        self.password_hash = row["password_hash"]

    @staticmethod
    def find_by_id(admin_id):
        with get_db() as conn:
            row = conn.execute("SELECT * FROM admins WHERE id = ?", (admin_id,)).fetchone()
        return Admin(row) if row else None

    @staticmethod
    def find_by_username(username):
        with get_db() as conn:
            row = conn.execute("SELECT * FROM admins WHERE username = ?", (username,)).fetchone()
        return Admin(row) if row else None


@login_manager.user_loader
def load_user(user_id):
    return Admin.find_by_id(user_id)


# =========================================================
# 4. TEMPLATE HTML LOGIN
# =========================================================

LOGIN_HTML = r"""
<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Pinkmart | Login Admin</title>
<style>
* { box-sizing: border-box; }
body {
    margin: 0; min-height: 100vh; display: grid; place-items: center; padding: 28px;
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; color: #4A4A4A;
    background: linear-gradient(135deg, #FFD1DC 0%, #AEC6CF 50%, #E6E6FA 100%);
}
.container {
    width: min(100%, 960px); display: grid; grid-template-columns: 1fr 1fr;
    overflow: hidden; border-radius: 24px; background: #FFFFFF; box-shadow: 0 20px 50px rgba(0,0,0,0.1);
}
.hero {
    padding: 48px; background: linear-gradient(145deg, #FFC0CB, #E6E6FA);
    display: flex; flex-direction: column; justify-content: center;
}
.logo { color: #D87093; font-size: 32px; font-weight: 900; letter-spacing: 2px; }
.tag { margin-top: 30px; color: #6A5ACD; font-size: 11px; font-weight: 800; letter-spacing: 2px; }
.hero h1 { font-size: clamp(28px, 4vw, 38px); color: #4A4A4A; line-height: 1.2; margin: 16px 0; }
.hero p { color: #666; line-height: 1.8; font-size: 14px; }
.form-panel { padding: 48px; background: #FFFFFF; }
.eyebrow { color: #708090; letter-spacing: 2px; font-size: 11px; font-weight: 800; }
h2 { margin: 14px 0 8px; font-size: 28px; color: #333; }
.subtitle { color: #777; font-size: 14px; margin-bottom: 28px; }
label { display: block; margin: 16px 0 6px; font-size: 13px; color: #555; font-weight: 600; }
input {
    display: block; width: 100%; padding: 12px 14px; color: #333;
    background: #F4F6F9; border: 1px solid #DCDCDC; border-radius: 10px; outline: none; font-size: 14px;
}
input:focus { border-color: #B0E0E6; box-shadow: 0 0 0 3px rgba(176,224,230,0.5); }
button {
    width: 100%; margin-top: 25px; padding: 14px; color: white;
    background: linear-gradient(90deg, #FFB6C1, #9370DB); border: 0;
    border-radius: 10px; font-weight: 700; cursor: pointer; transition: all .3s;
}
button:hover { opacity: 0.9; transform: translateY(-1px); }
.flash { color:#721c24; background: #f8d7da; padding: 10px; border-radius: 8px; font-size: 13px; margin-bottom: 15px; text-align: center; }
.footer { margin-top: 30px; color: #AAA; font-size: 12px; }
@media (max-width: 720px) { .container { grid-template-columns: 1fr; } .hero, .form-panel { padding: 30px; } }
</style>
</head>
<body>
<div class="container">
    <section class="hero">
        <div class="logo">PINKMART</div>
        <div class="tag">SYSTEM PENJUALAN</div>
        <h1>Selamat Datang<br>di Web Admin.</h1>
        <p>Portal administrator Web Admin Pinkmart untuk kelola transaksi, jenis barang, kategori, pelanggan, dan laporan penjualan.</p>
    </section>

    <section class="form-panel">
        <div class="eyebrow">WELCOME BACK</div>
        <h2>Login Admin</h2>
        <div class="subtitle">Masukkan kredensial Anda untuk masuk ke sistem.</div>

        {% for message in get_flashed_messages() %}
            <div class="flash">{{ message }}</div>
        {% endfor %}

        <form method="POST">
            <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
            <label for="username">Username</label>
            <input id="username" name="username" type="text" placeholder="Masukkan username" required>

            <label for="password">Password</label>
            <input id="password" name="password" type="password" placeholder="Masukkan password" required>

            <button type="submit">Buka Web Penjualan &rarr;</button>
        </form>

        <div class="footer">Web Admin Pinkmart &copy; 2026</div>
    </section>
</div>
</body>
</html>
"""


# =========================================================
# 5. TEMPLATE BASE DASHBOARD MULTI-FITUR
# =========================================================

BASE_LAYOUT = r"""
<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Pinkmart | {{ title }}</title>
<style>
* { box-sizing: border-box; }
body { margin: 0; background: #F7F9FC; color: #4A4A4A; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }
.layout { display: flex; min-height: 100vh; }

.sidebar {
    width: 250px; flex-shrink: 0; padding: 28px 18px;
    background: linear-gradient(180deg, #FFB6C1 0%, #AEC6CF 100%);
    box-shadow: 2px 0 10px rgba(0,0,0,0.05);
}
.logo { color:#FFFFFF; font-size: 26px; font-weight: 900; letter-spacing: 2px; margin: 8px 12px 25px; text-shadow: 1px 1px 2px rgba(0,0,0,0.1); }
.menu-section-title { font-size: 11px; font-weight: 800; color: #4B0082; letter-spacing: 1.5px; margin: 18px 12px 6px; text-transform: uppercase; }

.nav {
    display: block; padding: 11px 14px; margin: 3px 0; border-radius: 10px; color: #333333;
    font-size: 13.5px; text-decoration: none; cursor: pointer; transition: all 0.2s; font-weight: 600;
}
.nav:hover { background: rgba(255, 255, 255, 0.4); color: #4B0082; }
.nav.active { color: #4B0082; background: #FFFFFF; font-weight: 700; box-shadow: 0 4px 10px rgba(0,0,0,0.05); }

main { width: 100%; max-width: 1500px; padding: 36px; margin: 0 auto; }
.topbar { display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 20px; }
.kicker { font-size: 11px; letter-spacing: 2px; font-weight: 800; color: #9370DB; }
h1 { font-size: 30px; margin: 8px 0; color: #333; }

.actions { display: flex; align-items: center; gap: 12px; }
.badge { color: #4B0082; background: #E6E6FA; border-radius: 99px; padding: 9px 14px; font-size: 12px; font-weight: 700; }
.btn-add-trx {
    color: white; background: linear-gradient(90deg, #FFB6C1, #9370DB);
    padding: 10px 16px; border-radius: 10px; text-decoration: none; font-size: 13px;
    font-weight: 700; border: none; cursor: pointer; transition: transform 0.2s;
}
.btn-add-trx:hover { transform: translateY(-1px); opacity: 0.95; }
.logout { color: white; background: #FF6B6B; padding: 10px 16px; border-radius: 10px; text-decoration: none; font-size: 13px; font-weight: 700; }

.cards { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 18px; margin: 28px 0; }
.card, .panel {
    background: #FFFFFF; border-radius: 16px; padding: 22px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.03); border: 1px solid #EAEAEA;
}
.card-label { font-size: 13px; font-weight: 700; color: #777; }
.value { font-size: 24px; font-weight: 800; margin: 12px 0 6px; color: #4B0082; }
.trend { font-size: 12px; color: #888; }

table { width: 100%; border-collapse: collapse; margin-top: 18px; }
th, td { text-align: left; padding: 14px 10px; border-bottom: 1px solid #F0F0F0; font-size: 13px; }
th { color: #6A5ACD; font-size: 11px; letter-spacing: 1px; font-weight: 800; }

.form-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-top: 15px; }
.form-group { display: flex; flex-direction: column; }
.form-group label { font-size: 12px; font-weight: 700; margin-bottom: 6px; color: #555; }
.form-group input, .form-group select, .form-group textarea { padding: 10px; border-radius: 8px; border: 1px solid #DDD; font-size: 13px; outline: none; }
.form-group input:focus, .form-group select:focus { border-color: #AEC6CF; }

.btn-submit { margin-top: 15px; padding: 12px 20px; background: #9370DB; color: white; border: none; border-radius: 8px; font-weight: bold; cursor: pointer; text-decoration: none; display: inline-block; }
.btn-submit:hover { background: #7B68EE; }
.btn-export { background: #77DD77; }
.btn-print { background: #AEC6CF; }

.filter-bar { display: flex; gap: 15px; align-items: center; margin-bottom: 20px; flex-wrap: wrap; }
.filter-group { display: flex; align-items: center; gap: 8px; }

.progress-bar { background: #F0F0F0; border-radius: 10px; height: 18px; width: 100%; overflow: hidden; margin-top: 8px; }
.progress-fill { background: linear-gradient(90deg, #AEC6CF, #9370DB); height: 100%; text-align: right; padding-right: 8px; color: white; font-size: 11px; line-height: 18px; font-weight: bold; }

.flash-msg { padding: 12px; background: #E6E6FA; color: #4B0082; border-radius: 8px; margin-bottom: 20px; font-size: 14px; font-weight: bold; }

.modal { display: none; position: fixed; z-index: 1000; left: 0; top: 0; width: 100%; height: 100%; overflow: auto; background-color: rgba(0,0,0,0.4); }
.modal-content { background-color: #FFFFFF; margin: 4% auto; padding: 28px; border-radius: 20px; width: 90%; max-width: 650px; box-shadow: 0 10px 30px rgba(0,0,0,0.15); }
.close { color: #aaa; float: right; font-size: 28px; font-weight: bold; cursor: pointer; }
.close:hover { color: #000; }

@media print {
    .sidebar, .topbar, .btn-submit, .filter-bar, form { display: none !important; }
    main { width: 100%; padding: 0; }
    .panel { border: none; box-shadow: none; }
}
@media (max-width: 1024px) { .cards { grid-template-columns: repeat(2, 1fr); } }
@media (max-width: 850px) { .sidebar { display: none; } main { padding: 22px; } .cards { grid-template-columns: 1fr; } .form-grid { grid-template-columns: 1fr; } }
</style>
</head>

<body>
<div class="layout">
    <aside class="sidebar">
        <div class="logo">PINKMART</div>

        <div class="menu-section-title">MENU UTAMA</div>
        <a href="{{ url_for('dashboard') }}" class="nav {% if active_page == 'overview' %}active{% endif %}">&#9638; &nbsp; Overview & Analysis</a>
        <a href="{{ url_for('barang') }}" class="nav {% if active_page == 'barang' %}active{% endif %}">&#9639; &nbsp; Kelola Produk</a>
        <a href="{{ url_for('kategori') }}" class="nav {% if active_page == 'kategori' %}active{% endif %}">&#9776; &nbsp; Kelola Kategori / Jenis</a>
        <a href="{{ url_for('penjualan') }}" class="nav {% if active_page == 'penjualan' %}active{% endif %}">&#9672; &nbsp; Kelola Pesanan</a>
        <a href="{{ url_for('pelanggan') }}" class="nav {% if active_page == 'pelanggan' %}active{% endif %}">&#9818; &nbsp; Kelola Pelanggan</a>
        <a href="{{ url_for('laporan') }}" class="nav {% if active_page == 'laporan' %}active{% endif %}">&#128462; &nbsp; Laporan Penjualan</a>

        <div class="menu-section-title" style="margin-top: 20px;">SISTEM</div>
        <a href="{{ url_for('settings') }}" class="nav {% if active_page == 'settings' %}active{% endif %}">&#9881; &nbsp; Sistem Pengaturan</a>
        <a href="{{ url_for('akun_web') }}" class="nav {% if active_page == 'akun_web' %}active{% endif %}">&#9813; &nbsp; Akun Web</a>
    </aside>

    <main>
        <div class="topbar">
            <div>
                <div class="kicker">ADMIN WORKSPACE / {{ kicker }}</div>
                <h1>{{ title }}</h1>
                <div style="color: #777; font-size: 14px;">{{ description }}</div>
            </div>
            <div class="actions">
                <span class="badge">&#9679; {{ app_settings.nama_aplikasi }} ({{ app_settings.status_portal }})</span>
                <button onclick="openModal()" class="btn-add-trx">+ Tambah Transaksi</button>
                <a class="logout" href="{{ url_for('logout') }}">Logout</a>
            </div>
        </div>

        {% for message in get_flashed_messages() %}
            <div class="flash-msg" style="margin-top:20px;">{{ message }}</div>
        {% endfor %}

        <!-- 1. MENU OVERVIEW & ANALYSIS -->
        {% if active_page == 'overview' %}
        <section class="cards">
            <div class="card">
                <div class="card-label">Total Produk</div>
                <div class="value">{{ total_barang }} Item</div>
                <div class="trend">Variasi Barang Terdaftar</div>
            </div>
            <div class="card">
                <div class="card-label">Total Pelanggan</div>
                <div class="value">{{ total_customers }}</div>
                <div class="trend">Pelanggan Terdaftar</div>
            </div>
            <div class="card">
                <div class="card-label">Total Transaksi</div>
                <div class="value">{{ total_trx }}</div>
                <div class="trend">Keseluruhan Pesanan</div>
            </div>
            <div class="card">
                <div class="card-label">Total Pendapatan</div>
                <div class="value">{{ app_settings.mata_uang_default }} {{ "{:,.0f}".format(total_omset) }}</div>
                <div class="trend">Akumulasi Omset</div>
            </div>
        </section>

        <!-- KELOMPOK JENIS PRODUK DI DASHBOARD -->
        <section class="panel" style="margin-bottom: 25px;">
            <h2 style="margin-top:0;">Daftar Jenis / Kategori Produk yang Dijual</h2>
            <p style="font-size: 13px; color: #666;">Berikut adalah kategori barang dan produk yang tersedia di Web Admin Pinkmart:</p>
            <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-top: 15px;">
                {% for k in categories %}
                <div style="background: #F4F6F9; border-radius: 12px; padding: 15px; border-left: 4px solid #9370DB;">
                    <div style="font-weight: bold; color: #4B0082; font-size: 15px;">{{ k.nama_kategori }}</div>
                    <div style="font-size: 12px; color: #666; margin-top: 5px;">{{ k.deskripsi }}</div>
                </div>
                {% endfor %}
            </div>
        </section>

        <section class="panel" style="margin-bottom: 25px;">
            <h2 style="margin-top:0;">Analitik Ringkasan Penjualan</h2>
            <p style="font-size: 13px; color: #666;">Performa pesanan selesai dan rasio pesanan pending saat ini.</p>

            <div style="margin-top: 15px;">
                <label style="font-size: 12px; font-weight: bold;">Tingkat Keberhasilan Pesanan (Status Selesai):</label>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: {{ success_rate }}%;">{{ "{:.1f}".format(success_rate) }}%</div>
                </div>
            </div>

            <div style="margin-top: 15px;">
                <label style="font-size: 12px; font-weight: bold;">Rasio Pesanan Pending ({{ pending_trx }} Pesanan Pending):</label>
                <div class="progress-bar">
                    <div class="progress-fill" style="width: {{ pending_rate }}%; background: linear-gradient(90deg, #FFB6C1, #FF6B6B);">{{ "{:.1f}".format(pending_rate) }}%</div>
                </div>
            </div>
        </section>

        <section class="panel">
            <h2 style="margin-top:0;">Transaksi Penjualan Terbaru</h2>
            <table>
                <thead>
                    <tr>
                        <th>KODE TRX</th>
                        <th>NAMA PELANGGAN</th>
                        <th>JENIS / KATEGORI</th>
                        <th>PRODUK</th>
                        <th>TOTAL HARGA</th>
                        <th>STATUS</th>
                    </tr>
                </thead>
                <tbody>
                    {% for row in recent_sales %}
                    <tr>
                        <td><b>{{ row.kode_trx }}</b></td>
                        <td>{{ row.nama_pelanggan }}</td>
                        <td>{{ row.kategori }}</td>
                        <td>{{ row.produk }}</td>
                        <td>{{ app_settings.mata_uang_default }} {{ "{:,.0f}".format(row.total_harga) }}</td>
                        <td><b>{{ row.status }}</b></td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- 2. KELOLA PRODUK -->
        {% elif active_page == 'barang' %}
        <section class="cards">
            <div class="card">
                <div class="card-label">Total Produk</div>
                <div class="value">{{ total_barang }} Item</div>
                <div class="trend">Katalog Produk</div>
            </div>
            <div class="card">
                <div class="card-label">Total Stok Terkumpul</div>
                <div class="value">{{ total_stok }} Pcs</div>
                <div class="trend">Persediaan Produk</div>
            </div>
        </section>

        <section class="panel" style="margin-bottom: 25px;">
            <h2 style="margin-top:0;">Tambah Produk Baru</h2>
            <form method="POST" action="{{ url_for('tambah_barang') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Nama Produk</label>
                        <input type="text" name="nama_barang" required placeholder="Contoh: Headphones Bluetooth">
                    </div>
                    <div class="form-group">
                        <label>Jenis / Kategori Produk</label>
                        <select name="kategori">
                            {% for k in categories %}
                            <option value="{{ k.nama_kategori }}">{{ k.nama_kategori }}</option>
                            {% endfor %}
                        </select>
                    </div>
                    <div class="form-group">
                        <label>Harga Satuan ({{ app_settings.mata_uang_default }})</label>
                        <input type="number" name="harga" required placeholder="Contoh: 350000">
                    </div>
                    <div class="form-group">
                        <label>Jumlah Stok</label>
                        <input type="number" name="stok" required value="10" min="0">
                    </div>
                </div>
                <button type="submit" class="btn-submit">+ Simpan Data Produk</button>
            </form>
        </section>

        <section class="panel">
            <h2 style="margin-top:0;">Daftar Katalog Produk</h2>
            <table>
                <thead>
                    <tr>
                        <th>KODE</th>
                        <th>NAMA PRODUK</th>
                        <th>JENIS / KATEGORI</th>
                        <th>HARGA SATUAN</th>
                        <th>STOK</th>
                    </tr>
                </thead>
                <tbody>
                    {% for b in items %}
                    <tr>
                        <td><b>{{ b.kode_barang }}</b></td>
                        <td>{{ b.nama_barang }}</td>
                        <td>{{ b.kategori }}</td>
                        <td>{{ app_settings.mata_uang_default }} {{ "{:,.0f}".format(b.harga) }}</td>
                        <td><b>{{ b.stok }} Pcs</b></td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- 3. KELOLA KATEGORI / JENIS PRODUK -->
        {% elif active_page == 'kategori' %}
        <section class="panel" style="margin-bottom: 25px;">
            <h2 style="margin-top:0;">Tambah Jenis / Kategori Produk Baru</h2>
            <form method="POST" action="{{ url_for('tambah_kategori') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Nama Jenis / Kategori</label>
                        <input type="text" name="nama_kategori" required placeholder="Contoh: Perabotan">
                    </div>
                    <div class="form-group">
                        <label>Deskripsi Jenis Produk</label>
                        <input type="text" name="deskripsi" required placeholder="Contoh: Peralatan rumah tangga">
                    </div>
                </div>
                <button type="submit" class="btn-submit">+ Simpan Kategori</button>
            </form>
        </section>

        <section class="panel">
            <h2 style="margin-top:0;">Daftar Jenis / Kategori Produk yang Dijual</h2>
            <table>
                <thead>
                    <tr>
                        <th>ID</th>
                        <th>NAMA JENIS / KATEGORI</th>
                        <th>DESKRIPSI HAFALAN / KETERANGAN</th>
                    </tr>
                </thead>
                <tbody>
                    {% for k in categories %}
                    <tr>
                        <td><b>#{{ k.id }}</b></td>
                        <td><b>{{ k.nama_kategori }}</b></td>
                        <td>{{ k.deskripsi }}</td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- 4. KELOLA PESANAN -->
        {% elif active_page == 'penjualan' %}
        <section class="cards">
            <div class="card">
                <div class="card-label">Total Transaksi</div>
                <div class="value">{{ total_trx }}</div>
                <div class="trend">Keseluruhan Pesanan</div>
            </div>
            <div class="card">
                <div class="card-label">Pesanan Pending</div>
                <div class="value">{{ pending_trx }}</div>
                <div class="trend">Dalam Proses</div>
            </div>
            <div class="card">
                <div class="card-label">Total Pendapatan</div>
                <div class="value">{{ app_settings.mata_uang_default }} {{ "{:,.0f}".format(total_omset) }}</div>
                <div class="trend">Akumulasi Penjualan</div>
            </div>
        </section>

        <section class="panel">
            <h2 style="margin-top:0;">Daftar & Status Pesanan Penjualan</h2>
            <form method="GET" action="{{ url_for('penjualan') }}" class="filter-bar" style="margin-top: 15px;">
                <div class="filter-group">
                    <label style="font-size:12px; font-weight:bold;">Status:</label>
                    <select name="status" onchange="this.form.submit()" style="padding: 6px 10px; border-radius: 6px;">
                        <option value="Semua" {% if selected_status == 'Semua' %}selected{% endif %}>Semua Status</option>
                        <option value="Selesai" {% if selected_status == 'Selesai' %}selected{% endif %}>Selesai</option>
                        <option value="Pending" {% if selected_status == 'Pending' %}selected{% endif %}>Pending</option>
                        <option value="Batal" {% if selected_status == 'Batal' %}selected{% endif %}>Batal</option>
                    </select>
                </div>
                <div class="filter-group">
                    <label style="font-size:12px; font-weight:bold;">Jenis / Kategori:</label>
                    <select name="kategori" onchange="this.form.submit()" style="padding: 6px 10px; border-radius: 6px;">
                        <option value="Semua" {% if selected_kategori == 'Semua' %}selected{% endif %}>Semua Kategori</option>
                        {% for k in categories %}
                        <option value="{{ k.nama_kategori }}" {% if selected_kategori == k.nama_kategori %}selected{% endif %}>{{ k.nama_kategori }}</option>
                        {% endfor %}
                    </select>
                </div>
            </form>

            <table>
                <thead>
                    <tr>
                        <th>KODE TRX</th>
                        <th>TANGGAL</th>
                        <th>NAMA PELANGGAN</th>
                        <th>JENIS / KATEGORI</th>
                        <th>PRODUK</th>
                        <th>QTY</th>
                        <th>TOTAL HARGA</th>
                        <th>STATUS</th>
                    </tr>
                </thead>
                <tbody>
                    {% for row in sales %}
                    <tr>
                        <td><b>{{ row.kode_trx }}</b></td>
                        <td>{{ row.tanggal }}</td>
                        <td>{{ row.nama_pelanggan }}</td>
                        <td>{{ row.kategori }}</td>
                        <td>{{ row.produk }}</td>
                        <td>{{ row.jumlah }}</td>
                        <td>{{ app_settings.mata_uang_default }} {{ "{:,.0f}".format(row.total_harga) }}</td>
                        <td><b>{{ row.status }}</b></td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- 5. KELOLA PELANGGAN -->
        {% elif active_page == 'pelanggan' %}
        <section class="cards">
            <div class="card">
                <div class="card-label">Total Pelanggan</div>
                <div class="value">{{ total_customers }}</div>
                <div class="trend">Pelanggan Terdaftar</div>
            </div>
            <div class="card">
                <div class="card-label">Pelanggan Setia</div>
                <div class="value">{{ loyal_customers }}</div>
                <div class="trend">Member VIP / Loyalty</div>
            </div>
        </section>

        <section class="panel" style="margin-bottom: 25px;">
            <h2 style="margin-top:0;">Tambah Data Pelanggan Baru</h2>
            <form method="POST" action="{{ url_for('tambah_pelanggan') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Nama Lengkap</label>
                        <input type="text" name="nama" required placeholder="Contoh: Gilang Faturahman">
                    </div>
                    <div class="form-group">
                        <label>Alamat Email</label>
                        <input type="email" name="email" required placeholder="Contoh: gilang@gmail.com">
                    </div>
                    <div class="form-group">
                        <label>No. Telepon / WA</label>
                        <input type="text" name="telepon" required placeholder="Contoh: 08123456789">
                    </div>
                    <div class="form-group">
                        <label>Status Pelanggan</label>
                        <select name="status_pelanggan">
                            <option value="Reguler">Reguler</option>
                            <option value="Pelanggan Setia">Pelanggan Setia</option>
                        </select>
                    </div>
                    <div class="form-group" style="grid-column: span 2;">
                        <label>Alamat</label>
                        <input type="text" name="alamat" required placeholder="Contoh: Tangerang, Banten">
                    </div>
                </div>
                <button type="submit" class="btn-submit">+ Tambah Pelanggan</button>
            </form>
        </section>

        <section class="panel">
            <h2 style="margin-top:0;">Manajemen Data Pelanggan</h2>
            <table>
                <thead>
                    <tr>
                        <th>KODE</th>
                        <th>NAMA PELANGGAN</th>
                        <th>EMAIL</th>
                        <th>TELEPON</th>
                        <th>ALAMAT</th>
                        <th>STATUS</th>
                        <th>TANGGAL DAFTAR</th>
                    </tr>
                </thead>
                <tbody>
                    {% for c in customers %}
                    <tr>
                        <td><b>{{ c.kode_pelanggan }}</b></td>
                        <td>{{ c.nama }}</td>
                        <td>{{ c.email }}</td>
                        <td>{{ c.telepon }}</td>
                        <td>{{ c.alamat }}</td>
                        <td><b>{{ c.status_pelanggan }}</b></td>
                        <td>{{ c.tanggal_daftar }}</td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- 6. LAPORAN PENJUALAN -->
        {% elif active_page == 'laporan' %}
        <section class="panel">
            <div style="display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 10px;">
                <h2 style="margin:0;">Pusat Laporan Penjualan</h2>
                <div class="actions">
                    <a href="{{ url_for('export_csv', status=selected_status, kategori=selected_kategori) }}" class="btn-submit btn-export">Export CSV</a>
                    <button onclick="window.print()" class="btn-submit btn-print">Print Laporan</button>
                </div>
            </div>

            <form method="GET" action="{{ url_for('laporan') }}" class="filter-bar" style="margin-top: 20px;">
                <div class="filter-group">
                    <label style="font-size:12px; font-weight:bold;">Filter Status:</label>
                    <select name="status" onchange="this.form.submit()" style="padding: 6px 10px; border-radius: 6px;">
                        <option value="Semua" {% if selected_status == 'Semua' %}selected{% endif %}>Semua Status</option>
                        <option value="Selesai" {% if selected_status == 'Selesai' %}selected{% endif %}>Selesai</option>
                        <option value="Pending" {% if selected_status == 'Pending' %}selected{% endif %}>Pending</option>
                        <option value="Batal" {% if selected_status == 'Batal' %}selected{% endif %}>Batal</option>
                    </select>
                </div>
                <div class="filter-group">
                    <label style="font-size:12px; font-weight:bold;">Filter Jenis / Kategori:</label>
                    <select name="kategori" onchange="this.form.submit()" style="padding: 6px 10px; border-radius: 6px;">
                        <option value="Semua" {% if selected_kategori == 'Semua' %}selected{% endif %}>Semua Kategori</option>
                        {% for k in categories %}
                        <option value="{{ k.nama_kategori }}" {% if selected_kategori == k.nama_kategori %}selected{% endif %}>{{ k.nama_kategori }}</option>
                        {% endfor %}
                    </select>
                </div>
            </form>

            <table>
                <thead>
                    <tr>
                        <th>KODE TRX</th>
                        <th>TANGGAL</th>
                        <th>NAMA PELANGGAN</th>
                        <th>JENIS / KATEGORI</th>
                        <th>PRODUK</th>
                        <th>QTY</th>
                        <th>TOTAL HARGA</th>
                        <th>STATUS</th>
                    </tr>
                </thead>
                <tbody>
                    {% for row in sales %}
                    <tr>
                        <td><b>{{ row.kode_trx }}</b></td>
                        <td>{{ row.tanggal }}</td>
                        <td>{{ row.nama_pelanggan }}</td>
                        <td>{{ row.kategori }}</td>
                        <td>{{ row.produk }}</td>
                        <td>{{ row.jumlah }}</td>
                        <td>{{ app_settings.mata_uang_default }} {{ "{:,.0f}".format(row.total_harga) }}</td>
                        <td><b>{{ row.status }}</b></td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>

        <!-- 7. SISTEM PENGATURAN -->
        {% elif active_page == 'settings' %}
        <section class="panel" style="margin-bottom:25px;">
            <h2 style="margin-top:0;">Pengaturan Sistem Portal</h2>
            <form method="POST" action="{{ url_for('simpan_settings') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Nama Web Portal</label>
                        <input type="text" name="nama_aplikasi" value="{{ app_settings.nama_aplikasi }}" required>
                    </div>
                    <div class="form-group">
                        <label>Versi Sistem</label>
                        <input type="text" name="versi" value="{{ app_settings.versi }}" required>
                    </div>
                    <div class="form-group">
                        <label>Mata Uang Default</label>
                        <select name="mata_uang_default">
                            <option value="Rp" {% if app_settings.mata_uang_default == 'Rp' %}selected{% endif %}>Rp (Rupiah)</option>
                            <option value="IDR" {% if app_settings.mata_uang_default == 'IDR' %}selected{% endif %}>IDR</option>
                            <option value="$" {% if app_settings.mata_uang_default == '$' %}selected{% endif %}>USD ($)</option>
                        </select>
                    </div>
                    <div class="form-group">
                        <label>Status Portal Web</label>
                        <select name="status_portal">
                            <option value="Aktif" {% if app_settings.status_portal == 'Aktif' %}selected{% endif %}>Aktif / Online</option>
                            <option value="Maintenance" {% if app_settings.status_portal == 'Maintenance' %}selected{% endif %}>Maintenance</option>
                        </select>
                    </div>
                    <div class="form-group" style="grid-column: span 2;">
                        <label>Footer Teks</label>
                        <input type="text" name="footer_text" value="{{ app_settings.footer_text }}" required>
                    </div>
                </div>
                <button type="submit" class="btn-submit">Simpan Pengaturan Aplikasi</button>
            </form>
        </section>

        <!-- 8. AKUN WEB (SEBAGAI UJUNG MENU SISTEM) -->
        {% elif active_page == 'akun_web' %}
        <section class="panel" style="margin-bottom: 25px;">
            <h2 style="margin-top:0;">Ubah Password Administrator Saya</h2>
            <form method="POST" action="{{ url_for('ubah_password') }}">
                <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
                <div class="form-grid">
                    <div class="form-group">
                        <label>Password Lama</label>
                        <input type="password" name="password_lama" required placeholder="Masukkan password lama">
                    </div>
                    <div class="form-group">
                        <label>Password Baru</label>
                        <input type="password" name="password_baru" required placeholder="Masukkan password baru">
                    </div>
                </div>
                <button type="submit" class="btn-submit" style="background:#FF6B6B;">Ubah Password</button>
            </form>
        </section>

        <section class="panel">
            <h2 style="margin-top:0;">Daftar Akun Administrator Web</h2>
            <table>
                <thead>
                    <tr>
                        <th>ID</th>
                        <th>USERNAME</th>
                        <th>STATUS AKSES</th>
                    </tr>
                </thead>
                <tbody>
                    {% for a in admins %}
                    <tr>
                        <td><b>#{{ a.id }}</b></td>
                        <td>{{ a.username }}</td>
                        <td><span style="color:#28a745; font-weight:bold;">Aktif (Full Access)</span></td>
                    </tr>
                    {% endfor %}
                </tbody>
            </table>
        </section>
        {% endif %}
    </main>
</div>

<!-- MODAL POPUP TRANSAKSI BARU -->
<div id="trxModal" class="modal">
    <div class="modal-content">
        <span class="close" onclick="closeModal()">&times;</span>
        <h2 style="margin-top:0; color:#4B0082;">Formulir Transaksi Penjualan Baru</h2>
        <form method="POST" action="{{ url_for('tambah_penjualan') }}">
            <input type="hidden" name="csrf_token" value="{{ csrf_token() }}">
            <div class="form-grid">
                <div class="form-group">
                    <label>Nama Pelanggan</label>
                    <input type="text" name="nama_pelanggan" required placeholder="Contoh: Gilang Faturahman">
                </div>
                <div class="form-group">
                    <label>Jenis / Kategori Produk</label>
                    <select name="kategori">
                        {% for k in categories %}
                        <option value="{{ k.nama_kategori }}">{{ k.nama_kategori }}</option>
                        {% endfor %}
                    </select>
                </div>
                <div class="form-group">
                    <label>Nama Produk / Barang</label>
                    <input type="text" name="produk" required placeholder="Contoh: Laptop Asus Zenbook">
                </div>
                <div class="form-group">
                    <label>Jumlah (Qty)</label>
                    <input type="number" name="jumlah" required value="1" min="1">
                </div>
                <div class="form-group">
                    <label>Total Harga ({{ app_settings.mata_uang_default }})</label>
                    <input type="number" name="total_harga" required placeholder="Contoh: 1500000">
                </div>
                <div class="form-group">
                    <label>Status Transaksi</label>
                    <select name="status">
                        <option value="Selesai">Selesai</option>
                        <option value="Pending">Pending</option>
                        <option value="Batal">Batal</option>
                    </select>
                </div>
            </div>
            <button type="submit" class="btn-submit" style="width:100%; margin-top:20px;">+ Simpan Transaksi Baru</button>
        </form>
    </div>
</div>

<script>
function openModal() { document.getElementById("trxModal").style.display = "block"; }
function closeModal() { document.getElementById("trxModal").style.display = "none"; }
window.onclick = function(event) {
    var modal = document.getElementById("trxModal");
    if (event.target == modal) { modal.style.display = "none"; }
}
</script>
</body>
</html>
"""


# =========================================================
# 6. ROUTE FLASK
# =========================================================

def get_settings():
    with get_db() as conn:
        return conn.execute("SELECT * FROM settings WHERE id = 1").fetchone()

def get_categories():
    with get_db() as conn:
        return conn.execute("SELECT * FROM kategori ORDER BY id ASC").fetchall()


@app.route("/", methods=["GET", "POST"])
def login():
    if current_user.is_authenticated:
        return redirect(url_for("dashboard"))

    if request.method == "POST":
        username = request.form.get("username", "").strip()
        password = request.form.get("password", "")

        admin = Admin.find_by_username(username)
        if admin and check_password_hash(admin.password_hash, password):
            login_user(admin)
            return redirect(url_for("dashboard"))

        flash("Username atau password salah.")

    return render_template_string(LOGIN_HTML)


# --- 1. OVERVIEW & ANALYSIS ---
@app.route("/dashboard")
@login_required
def dashboard():
    with get_db() as conn:
        recent_sales = conn.execute("SELECT * FROM penjualan ORDER BY id DESC LIMIT 5").fetchall()
        total_omset = conn.execute("SELECT SUM(total_harga) FROM penjualan").fetchone()[0] or 0
        total_trx = conn.execute("SELECT COUNT(id) FROM penjualan").fetchone()[0] or 0
        total_barang = conn.execute("SELECT COUNT(id) FROM barang").fetchone()[0] or 0
        total_customers = conn.execute("SELECT COUNT(id) FROM pelanggan").fetchone()[0] or 0
        pending_trx = conn.execute("SELECT COUNT(id) FROM penjualan WHERE status = 'Pending'").fetchone()[0] or 0
        success_trx = conn.execute("SELECT COUNT(id) FROM penjualan WHERE status = 'Selesai'").fetchone()[0] or 0

    success_rate = (success_trx / total_trx * 100) if total_trx > 0 else 0
    pending_rate = (pending_trx / total_trx * 100) if total_trx > 0 else 0

    return render_template_string(
        BASE_LAYOUT,
        active_page="overview",
        kicker="OVERVIEW & ANALYSIS",
        title="Dashboard Utama & Analisis",
        description="Ringkasan statistik total produk, pelanggan, transaksi, dan daftar jenis produk yang dijual.",
        recent_sales=recent_sales,
        total_omset=total_omset,
        total_trx=total_trx,
        total_barang=total_barang,
        total_customers=total_customers,
        pending_trx=pending_trx,
        success_rate=success_rate,
        pending_rate=pending_rate,
        categories=get_categories(),
        app_settings=get_settings()
    )


# --- 2. KELOLA PRODUK ---
@app.route("/barang")
@login_required
def barang():
    with get_db() as conn:
        items = conn.execute("SELECT * FROM barang ORDER BY id DESC").fetchall()
        total_barang = conn.execute("SELECT COUNT(id) FROM barang").fetchone()[0] or 0
        total_stok = conn.execute("SELECT SUM(stok) FROM barang").fetchone()[0] or 0

    return render_template_string(
        BASE_LAYOUT,
        active_page="barang",
        kicker="KELOLA PRODUK",
        title="Kelola Data Produk",
        description="Manajemen katalog produk dan persediaan stok barang.",
        items=items,
        total_barang=total_barang,
        total_stok=total_stok,
        categories=get_categories(),
        app_settings=get_settings()
    )


@app.route("/tambah-barang", methods=["POST"])
@login_required
def tambah_barang():
    nama_barang = request.form.get("nama_barang")
    kategori = request.form.get("kategori")
    harga = float(request.form.get("harga", 0))
    stok = int(request.form.get("stok", 0))

    with get_db() as conn:
        count = conn.execute("SELECT COUNT(id) FROM barang").fetchone()[0] + 1
        kode_barang = f"BRG-{count:03d}"
        conn.execute("""
            INSERT INTO barang (kode_barang, nama_barang, kategori, harga, stok)
            VALUES (?, ?, ?, ?, ?)
        """, (kode_barang, nama_barang, kategori, harga, stok))

    flash("Data Produk Berhasil Ditambahkan!")
    return redirect(url_for("barang"))


# --- 3. KELOLA KATEGORI / JENIS PRODUK ---
@app.route("/kategori")
@login_required
def kategori():
    return render_template_string(
        BASE_LAYOUT,
        active_page="kategori",
        kicker="KELOLA KATEGORI / JENIS",
        title="Daftar Jenis / Kategori Produk yang Dijual",
        description="Kelola kelompok dan pengelompokan jenis produk yang dijual.",
        categories=get_categories(),
        app_settings=get_settings()
    )


@app.route("/tambah-kategori", methods=["POST"])
@login_required
def tambah_kategori():
    nama_kategori = request.form.get("nama_kategori")
    deskripsi = request.form.get("deskripsi")

    try:
        with get_db() as conn:
            conn.execute("INSERT INTO kategori (nama_kategori, deskripsi) VALUES (?, ?)", (nama_kategori, deskripsi))
        flash("Jenis / Kategori Baru Berhasil Ditambahkan!")
    except sqlite3.IntegrityError:
        flash("Nama Kategori Sudah Ada!")

    return redirect(url_for("kategori"))


# --- 4. KELOLA PESANAN ---
@app.route("/penjualan")
@login_required
def penjualan():
    status = request.args.get("status", "Semua")
    kat = request.args.get("kategori", "Semua")

    query = "SELECT * FROM penjualan WHERE 1=1"
    params = []

    if status != "Semua":
        query += " AND status = ?"
        params.append(status)

    if kat != "Semua":
        query += " AND kategori = ?"
        params.append(kat)

    query += " ORDER BY id DESC"

    with get_db() as conn:
        sales = conn.execute(query, params).fetchall()
        total_omset = conn.execute("SELECT SUM(total_harga) FROM penjualan").fetchone()[0] or 0
        total_trx = conn.execute("SELECT COUNT(id) FROM penjualan").fetchone()[0] or 0
        pending_trx = conn.execute("SELECT COUNT(id) FROM penjualan WHERE status = 'Pending'").fetchone()[0] or 0

    return render_template_string(
        BASE_LAYOUT,
        active_page="penjualan",
        kicker="KELOLA PESANAN",
        title="Kelola Pesanan Penjualan",
        description="Kelola dan konfirmasi transaksi pesanan pelanggan.",
        sales=sales,
        total_omset=total_omset,
        total_trx=total_trx,
        pending_trx=pending_trx,
        selected_status=status,
        selected_kategori=kat,
        categories=get_categories(),
        app_settings=get_settings()
    )


@app.route("/tambah-penjualan", methods=["POST"])
@login_required
def tambah_penjualan():
    nama_pelanggan = request.form.get("nama_pelanggan")
    kategori_val = request.form.get("kategori", "Elektronik")
    produk = request.form.get("produk")
    jumlah = int(request.form.get("jumlah", 1))
    total_harga = float(request.form.get("total_harga", 0))
    status = request.form.get("status", "Selesai")
    tanggal = time.strftime("%Y-%m-%d")

    with get_db() as conn:
        count = conn.execute("SELECT COUNT(id) FROM penjualan").fetchone()[0] + 1
        kode_trx = f"#TRX-{count:03d}"
        conn.execute("""
            INSERT INTO penjualan (kode_trx, nama_pelanggan, kategori, produk, jumlah, total_harga, tanggal, status)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (kode_trx, nama_pelanggan, kategori_val, produk, jumlah, total_harga, tanggal, status))

    flash("Data Transaksi Berhasil Ditambahkan!")
    return redirect(request.referrer or url_for("penjualan"))


# --- 5. KELOLA PELANGGAN ---
@app.route("/pelanggan")
@login_required
def pelanggan():
    with get_db() as conn:
        customers = conn.execute("SELECT * FROM pelanggan ORDER BY id DESC").fetchall()
        total_customers = conn.execute("SELECT COUNT(id) FROM pelanggan").fetchone()[0] or 0
        loyal_customers = conn.execute("SELECT COUNT(id) FROM pelanggan WHERE status_pelanggan = 'Pelanggan Setia'").fetchone()[0] or 0

    return render_template_string(
        BASE_LAYOUT,
        active_page="pelanggan",
        kicker="KELOLA PELANGGAN",
        title="Kelola Data Pelanggan",
        description="Kelola direktori pelanggan dan status pelanggan setia.",
        customers=customers,
        total_customers=total_customers,
        loyal_customers=loyal_customers,
        categories=get_categories(),
        app_settings=get_settings()
    )


@app.route("/tambah-pelanggan", methods=["POST"])
@login_required
def tambah_pelanggan():
    nama = request.form.get("nama")
    email = request.form.get("email")
    telepon = request.form.get("telepon")
    alamat = request.form.get("alamat")
    status_pelanggan = request.form.get("status_pelanggan", "Reguler")
    tanggal_daftar = time.strftime("%Y-%m-%d")

    with get_db() as conn:
        count = conn.execute("SELECT COUNT(id) FROM pelanggan").fetchone()[0] + 1
        kode_pelanggan = f"CUST-{count:03d}"
        conn.execute("""
            INSERT INTO pelanggan (kode_pelanggan, nama, email, telepon, alamat, status_pelanggan, tanggal_daftar)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (kode_pelanggan, nama, email, telepon, alamat, status_pelanggan, tanggal_daftar))

    flash("Data Pelanggan Berhasil Ditambahkan!")
    return redirect(url_for("pelanggan"))


# --- 6. LAPORAN PENJUALAN ---
@app.route("/laporan")
@login_required
def laporan():
    status = request.args.get("status", "Semua")
    kat = request.args.get("kategori", "Semua")

    query = "SELECT * FROM penjualan WHERE 1=1"
    params = []

    if status != "Semua":
        query += " AND status = ?"
        params.append(status)

    if kat != "Semua":
        query += " AND kategori = ?"
        params.append(kat)

    query += " ORDER BY id DESC"

    with get_db() as conn:
        sales = conn.execute(query, params).fetchall()

    return render_template_string(
        BASE_LAYOUT,
        active_page="laporan",
        kicker="LAPORAN PENJUALAN",
        title="Laporan Penjualan",
        description="Cetak laporan penjualan atau unduh berkas CSV data transaksi.",
        sales=sales,
        selected_status=status,
        selected_kategori=kat,
        categories=get_categories(),
        app_settings=get_settings()
    )


@app.route("/export-csv")
@login_required
def export_csv():
    status = request.args.get("status", "Semua")
    kat = request.args.get("kategori", "Semua")

    query = "SELECT kode_trx, tanggal, nama_pelanggan, kategori, produk AS nama_barang, jumlah, total_harga, status FROM penjualan WHERE 1=1"
    params = []

    if status != "Semua":
        query += " AND status = ?"
        params.append(status)

    if kat != "Semua":
        query += " AND kategori = ?"
        params.append(kat)

    with get_db() as conn:
        df = pd.read_sql_query(query, conn, params=params)

    output_stream = io.BytesIO()
    df.to_csv(output_stream, index=False)
    output_stream.seek(0)

    return Response(
        output_stream.getvalue(),
        mimetype="text/csv",
        headers={"Content-Disposition": "attachment;filename=laporan_penjualan.csv"}
    )


# --- 7. SISTEM PENGATURAN ---
@app.route("/settings")
@login_required
def settings():
    return render_template_string(
        BASE_LAYOUT,
        active_page="settings",
        kicker="SISTEM PENGATURAN",
        title="Pengaturan Sistem",
        description="Pengaturan preferensi portal dan sistem web.",
        categories=get_categories(),
        app_settings=get_settings()
    )


@app.route("/simpan-settings", methods=["POST"])
@login_required
def simpan_settings():
    nama_aplikasi = request.form.get("nama_aplikasi")
    versi = request.form.get("versi")
    mata_uang_default = request.form.get("mata_uang_default")
    status_portal = request.form.get("status_portal")
    footer_text = request.form.get("footer_text")

    with get_db() as conn:
        conn.execute("""
            UPDATE settings
            SET nama_aplikasi = ?, versi = ?, mata_uang_default = ?, status_portal = ?, footer_text = ?
            WHERE id = 1
        """, (nama_aplikasi, versi, mata_uang_default, status_portal, footer_text))

    flash("Pengaturan Sistem Berhasil Diperbarui!")
    return redirect(url_for("settings"))


# --- 8. AKUN WEB (UJUNG MENU SISTEM) ---
@app.route("/akun-web")
@login_required
def akun_web():
    with get_db() as conn:
        admins = conn.execute("SELECT id, username FROM admins ORDER BY id ASC").fetchall()

    return render_template_string(
        BASE_LAYOUT,
        active_page="akun_web",
        kicker="AKUN WEB",
        title="Manajemen Akun Web Administrator",
        description="Kelola akun administrator dan ubah kata sandi sistem.",
        admins=admins,
        categories=get_categories(),
        app_settings=get_settings()
    )


@app.route("/ubah-password", methods=["POST"])
@login_required
def ubah_password():
    password_lama = request.form.get("password_lama")
    password_baru = request.form.get("password_baru")

    with get_db() as conn:
        admin_row = conn.execute("SELECT * FROM admins WHERE id = ?", (current_user.id,)).fetchone()
        if admin_row and check_password_hash(admin_row["password_hash"], password_lama):
            new_hash = generate_password_hash(password_baru)
            conn.execute("UPDATE admins SET password_hash = ? WHERE id = ?", (new_hash, current_user.id))
            flash("Password Admin Berhasil Diubah!")
        else:
            flash("Password lama salah!")

    return redirect(url_for("akun_web"))


@app.route("/logout")
@login_required
def logout():
    logout_user()
    return redirect(url_for("login"))


# =========================================================
# 7. MENJALANKAN SERVER COLAB
# =========================================================

def start_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

if not globals().get("URWEB_server_started", False):
    server_thread = threading.Thread(target=start_flask, daemon=True)
    server_thread.start()
    globals()["URWEB_server_started"] = True
    time.sleep(2)

proxy_url = output.eval_js("google.colab.kernel.proxyPort(5000)")

display(HTML(f"""
<div style="font-family:'Segoe UI',Arial,sans-serif;padding:20px;
background:#FFFFFF;border:1px solid #FFB6C1;border-radius:14px;
color:#4A4A4A;max-width:600px;margin-top:10px;box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
  <h3 style="margin-top:0;color:#9370DB">Pinkmart Admin Portal Active</h3>
  <p style="margin-bottom:15px;color:#666;">Server berhasil berjalan di Colab. Login dengan kredensial:</p>
  <ul style="margin-bottom:15px;color:#333;font-size:14px;">
    <li><b>Username:</b> pinkmart</li>
    <li><b>Password:</b> adminpinkmart</li>
  </ul>
  <a href="{proxy_url}" target="_blank"
     style="display:inline-block;padding:12px 24px;background:linear-gradient(90deg, #FFB6C1, #9370DB);
     color:white;text-decoration:none;border-radius:10px;font-weight:bold;
     box-shadow: 0 4px 12px rgba(147, 112, 219, 0.3);">
     Buka Web Admin Pinkmart Portal &#8599;
  </a>
</div>
"""))
import os
import threading
import time

# 1. Jalankan server Flask di background
def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

if not globals().get("server_started", False):
    thread = threading.Thread(target=run_flask, daemon=True)
    thread.start()
    globals()["server_started"] = True
    time.sleep(2)

# 2. Jalankan Localtunnel untuk menghasilkan URL Publik yang bisa dibuka di HP/Device Mana Pun
print("Membuat link publik...")
!npx localtunnel --port 5000 > url.txt 2>&1 &
time.sleep(3)

with open("url.txt") as f:
    print("\n==================================================")
    print("LINK WEBSITES BISA DI-AKSES DARI DEVICE MANA PUN:")
    print(f.read().strip())
    print("==================================================\n")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Membuat link publik...

LINK WEBSITES BISA DI-AKSES DARI DEVICE MANA PUN:
npm warn exec The following package was not found and will be installed: localtunnel@2.0.2

